# Developing MCP tools

This is a guided walkthrough of developing a data product that exposes MCP tools

## Project Layout

- [transform.py](example_mcp/transform.py): defines the data product's transformation code. This creates a snowflake table containing a list of banks
- [models.py](example_mcp/models.py): defines the semantic models that this data product transform produces.
- [__mcp__.py](example_mcp/__mcp__.py): defines the MCP function and request/response models. The MCP function queries the snowflake table containing a list of banks and returns this to the caller
- [requirements.txt](example_mcp/requirements.txt): python packages this data product depends on
- [spec.py](example_mcp/spec.py): defines the transform, inputs and outputs of this data product

## Prerequisites

- Ensure `example_mcp` data product source is available in this jupyter instance

In [ ]:
%%bash
# Create a python virtual environment with all the data products dependencies
# This is required for the nxd CLI to parse the project and build the deployment package

rm -rf .venv
uv venv --python 3.11
source .venv/bin/activate
uv pip install -r example_mcp/requirements.txt

## Launch Data Product

The following command launches the data product containing the MCP server

In [ ]:
%%bash

source .venv/bin/activate
nxd launch --dir example_mcp

In [ ]:
!nxd describe data-product example-mcp-server-demo

## Visit Data Product

The data product's page can be viewed inside NextData at: https://app.demo.trynxd.com/data-products/example-mcp-server-demo

Note that:
- The MODELS tab shows `banks` model - this is snowflake table created in the transform function
- The API FUNCTIONS tab shows the `get_banks` MCP tool - this is defined in [__mcp__.py](example_mcp/__mcp__.py) module
- The OUTPUTs tab shows the output ports of this data product

### Get MCP API Details

Click on `mcp-api` in the OUTPUTs tab. This links to a documentation page at: https://app.demo.trynxd.com/data-products/example-mcp-server-demo/api-outputs/mcp-api?tab=documentation

- The Functions tab shows a list of tools that this server exposes
- The Documentation tab shows more information configure your agents to access these tools
- The Access tab shows the URL used to access this tool

## Connect to MCP tool

1. Create a personal access token with the command below

In [ ]:
!nxd create personal-access-token --name my-personal-access-token --expires P30D

2. Add the following configuration to your agent's configuration file:

```json
{
  "mcpServers": {
    "example-mcp-server": {
      "command": "npx",
      "args": ["-y", "mcp-remote@latest", "https://dp.demo.trynxd.com/example-mcp-server-demo/rpcs/mcp-api/mcp/", "--header", "x-nextdata-token:<YOUR_PERSONAL_ACCESS_TOKEN>"]
    }
  }
}
```

3. Restart Claude to allow your agent to apply the new configuration settings

## Interact with MCP tool

Try the prompt: 'Get me a list of banks from the example-mcp-server data product'.

This should return a list containing: `Westpac, ANZ and Macquarie`

## (Optional) Extend the MCP tool

The MCP tool `get_banks` currently returns a list of banks from a `BANKS` snowflake table. Extend this table to include other Australian banks such as:
- Commonwealth Bank
- National Australia Bank
- Bank of Queensland

HINT: 
- Modify [transform.py](example_mcp/transform.py) to include these other Australian banks
- Re-Deploy the data product
- Use the same prompt to check that the response now includes the other banks